In [5]:
# ==============================================================================
# 2026 FIFA World Cup - Ablation Study V4 (8 Combinations + Safe Tournament Engine)
# [Upgrade 1] Dynamic K-factor (친선/본선 가중치 분리)
# [Upgrade 2] Opponent-Adjusted Form (상대 체급 보정 승점)
# [Upgrade 3] Multi-Alpha EMA (지수 이동 평균 폼)
# ==============================================================================
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from scipy.stats import poisson
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from itertools import product
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# 1. Advanced Elo Class (Dynamic K 적용 가능 구조)
# ---------------------------------------------------------
class AdvancedElo:
    def __init__(self, base=1500):
        self.ratings = {}
        self.base = base

    def get_rating(self, team):
        return self.ratings.get(team, self.base)

    def expected_result(self, loc, awc):
        dr = loc - awc
        return 1 / (10 ** (-dr / 400) + 1)

    def update_rating(self, home_team, away_team, home_goals, away_goals, k_factor):
        home_rating = self.get_rating(home_team)
        away_rating = self.get_rating(away_team)

        goal_diff = abs(home_goals - away_goals)
        if goal_diff <= 1: G = 1.0
        elif goal_diff == 2: G = 1.5
        else: G = (11 + goal_diff) / 8.0

        home_expected = self.expected_result(home_rating + 100, away_rating)
        away_expected = self.expected_result(away_rating, home_rating + 100)

        if home_goals > away_goals: home_actual, away_actual = 1, 0
        elif home_goals < away_goals: home_actual, away_actual = 0, 1
        else: home_actual, away_actual = 0.5, 0.5

        self.ratings[home_team] = home_rating + k_factor * G * (home_actual - home_expected)
        self.ratings[away_team] = away_rating + k_factor * G * (away_actual - away_expected)

# ---------------------------------------------------------
# 2. LightGBM 최적화
# ---------------------------------------------------------
def optimize_and_train(X, y, n_trials=10):
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    def objective(trial):
        params = {
            'objective': 'poisson', 'metric': 'poisson',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'feature_pre_filter': False, 'verbose': -1, 'seed': 42
        }
        gbm = lgb.train(params, train_data, valid_sets=[valid_data], num_boost_round=300,
                        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)])
        return gbm.best_score['valid_0']['poisson']

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params
    best_params.update({'objective': 'poisson', 'feature_pre_filter': False, 'verbose': -1, 'seed': 42})
    return lgb.train(best_params, lgb.Dataset(X, label=y), num_boost_round=200)

# ---------------------------------------------------------
# 3. 우직한 정답지 시뮬레이션
# ---------------------------------------------------------
def simulate_match_deterministic(lambda_home, lambda_away, n_simulations=5000):
    home_goals_sim = poisson.rvs(mu=lambda_home, size=n_simulations)
    away_goals_sim = poisson.rvs(mu=lambda_away, size=n_simulations)
    
    home_wins = np.sum(home_goals_sim > away_goals_sim)
    draws = np.sum(home_goals_sim == away_goals_sim)
    away_wins = np.sum(home_goals_sim < away_goals_sim)
    
    prob_h = home_wins / n_simulations
    prob_d = draws / n_simulations
    prob_a = away_wins / n_simulations
    
    scores = [f"{h}-{a}" for h, a in zip(home_goals_sim, away_goals_sim)]
    most_frequent_score = max(set(scores), key=scores.count)
    pred_h, pred_a = map(int, most_frequent_score.split('-'))
    
    return prob_h, prob_d, prob_a, pred_h, pred_a

# ---------------------------------------------------------
# 4. 메인 실행 파이프라인
# ---------------------------------------------------------
if __name__ == "__main__":
    print("🚀 월드컵 고도화 이론 8종 세트 및 토너먼트 시뮬레이션 시작...\n")
    
    # [수정 완료] 폴더 스크린샷 내 실제 파일명 반영
    RESULTS_FILE = "historical_results.csv"
    MAPPING_FILE = "all_matched_country_v2.csv"
    RULES_FILE = "third_place_assignments_2026.csv"
    
    for f in [RESULTS_FILE, MAPPING_FILE, RULES_FILE]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"❌ 로컬 폴더에 '{f}' 파일이 존재하지 않습니다. 파일명을 재확인해주세요.")
            
    # 축구 데이터 특수문자 깨짐 방지를 위해 latin1 사용
    df_raw = pd.read_csv(RESULTS_FILE, encoding='latin1')
    df_raw['home_score'] = pd.to_numeric(df_raw['home_score'], errors='coerce')
    df_raw['away_score'] = pd.to_numeric(df_raw['away_score'], errors='coerce')
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values('date').reset_index(drop=True)
    df_raw['is_neutral'] = df_raw['neutral'].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)

    # 이름 교정
    df_map = pd.read_csv(MAPPING_FILE, encoding='utf-8-sig')
    name_dict = dict(zip(df_map['results_name'], df_map['최종_매핑명']))
    extra_corrections = {
        'Cura?ao': 'Curaçao', 'Curacao': 'Curaçao', 'DR Congo': 'Congo DR', 
        'Cape Verde': 'Cape Verde Islands', 'Czech Republic': 'Czechia',
        'Bosnia and Herzegovina': 'Bosnia-Herzegovina', 'Bosnia': 'Bosnia-Herzegovina'
    }
    name_dict.update(extra_corrections)
    df_raw['home_team'] = df_raw['home_team'].replace(name_dict)
    df_raw['away_team'] = df_raw['away_team'].replace(name_dict)

    # 조합(Combinations): (DynamicK, AdjForm, EMA)
    combinations = list(product([False, True], repeat=3))
    
    print("🧪 [실험 시작] 8개의 조합별 시뮬레이션을 순차적으로 구동합니다.\n" + "="*60)
    
    for use_dyn_k, use_adj_f, use_ema in combinations:
        tag_k = "O" if use_dyn_k else "X"
        tag_f = "O" if use_adj_f else "X"
        tag_e = "O" if use_ema else "X"
        filename = f"sub_DynK_{tag_k}_AdjF_{tag_f}_EMA_{tag_e}.csv"
        
        print(f"\n⚙️ 진행 중인 조합: Dynamic K({tag_k}) | Adj Form({tag_f}) | EMA({tag_e})")
        
        df = df_raw.copy()
        elo_system = AdvancedElo()
        team_stats = {}
        
        f_home_elo, f_away_elo = [], []
        f_h_sma_s, f_h_sma_c, f_a_sma_s, f_a_sma_c = [], [], [], []
        f_h_form, f_a_form = [], []
        f_h_ema_s, f_h_ema_c, f_a_ema_s, f_a_ema_c = [], [], [], []
        
        ALPHA1 = 0.2

        for idx, row in df.iterrows():
            ht, at = row['home_team'], row['away_team']
            trn = str(row['tournament'])
            
            hr = elo_system.get_rating(ht)
            ar = elo_system.get_rating(at)
            f_home_elo.append(hr); f_away_elo.append(ar)
            
            for t in (ht, at):
                if t not in team_stats:
                    team_stats[t] = {'scored':[], 'conceded':[], 'pts':[], 'ema_s':1.0, 'ema_c':1.0}
                    
            h_s = team_stats[ht]
            f_h_sma_s.append(np.mean(h_s['scored'][-10:]) if h_s['scored'] else 1.0)
            f_h_sma_c.append(np.mean(h_s['conceded'][-10:]) if h_s['conceded'] else 1.0)
            f_h_form.append(sum(h_s['pts'][-5:]) if h_s['pts'] else 0.0)
            f_h_ema_s.append(h_s['ema_s']); f_h_ema_c.append(h_s['ema_c'])
            
            a_s = team_stats[at]
            f_a_sma_s.append(np.mean(a_s['scored'][-10:]) if a_s['scored'] else 1.0)
            f_a_sma_c.append(np.mean(a_s['conceded'][-10:]) if a_s['conceded'] else 1.0)
            f_a_form.append(sum(a_s['pts'][-5:]) if a_s['pts'] else 0.0)
            f_a_ema_s.append(a_s['ema_s']); f_a_ema_c.append(a_s['ema_c'])
            
            if pd.notna(row['home_score']) and pd.notna(row['away_score']):
                hs, ast = float(row['home_score']), float(row['away_score'])
                
                if use_dyn_k:
                    if 'Friendly' in trn: k_factor = 10
                    elif 'FIFA World Cup' in trn and 'qualification' not in trn.lower(): k_factor = 40
                    else: k_factor = 20
                else:
                    k_factor = 20
                
                elo_system.update_rating(ht, at, hs, ast, k_factor)
                
                base_h_pts = 3 if hs > ast else (0 if hs < ast else 1)
                base_a_pts = 0 if hs > ast else (3 if hs < ast else 1)
                
                if use_adj_f:
                    h_pts = base_h_pts * (ar / 1500.0)
                    a_pts = base_a_pts * (hr / 1500.0)
                else:
                    h_pts, a_pts = base_h_pts, base_a_pts
                    
                h_s['scored'].append(hs); h_s['conceded'].append(ast); h_s['pts'].append(h_pts)
                a_s['scored'].append(ast); a_s['conceded'].append(hs); a_s['pts'].append(a_pts)
                
                if use_ema:
                    h_s['ema_s'] = ALPHA1 * hs + (1-ALPHA1) * h_s['ema_s']
                    h_s['ema_c'] = ALPHA1 * ast + (1-ALPHA1) * h_s['ema_c']
                    a_s['ema_s'] = ALPHA1 * ast + (1-ALPHA1) * a_s['ema_s']
                    a_s['ema_c'] = ALPHA1 * hs + (1-ALPHA1) * a_s['ema_c']

        df['home_elo'] = f_home_elo; df['away_elo'] = f_away_elo; df['elo_diff'] = df['home_elo'] - df['away_elo']
        df['h_sma_s'] = f_h_sma_s; df['h_sma_c'] = f_h_sma_c; df['a_sma_s'] = f_a_sma_s; df['a_sma_c'] = f_a_sma_c; df['h_form'] = f_h_form; df['a_form'] = f_a_form
        df['h_ema_s'] = f_h_ema_s; df['h_ema_c'] = f_h_ema_c; df['a_ema_s'] = f_a_ema_s; df['a_ema_c'] = f_a_ema_c

        features = ['home_elo', 'away_elo', 'elo_diff', 'is_neutral', 'h_form', 'a_form']
        if use_ema:
            features += ['h_ema_s', 'h_ema_c', 'a_ema_s', 'a_ema_c']
        else:
            features += ['h_sma_s', 'h_sma_c', 'a_sma_s', 'a_sma_c']

        mask_2026_wc = (df['date'].dt.year == 2026) & (df['tournament'] == 'FIFA World Cup')
        df_train = df[(~mask_2026_wc) & (df['date'].dt.year >= 2010)].dropna(subset=['home_score', 'away_score']).copy()
        df_test = df[mask_2026_wc].copy()
        
        if len(df_test) == 0:
            print("⚠️ 데이터셋 내에 2026년 타겟 경기(조별리그)가 발견되지 않았습니다. 데이터를 확인해 주세요.")
            continue

        lgb_home = optimize_and_train(df_train[features], df_train['home_score'], n_trials=5)
        lgb_away = optimize_and_train(df_train[features], df_train['away_score'], n_trials=5)
        
        # 2. 조별리그 예측 연산 및 승점 산정
        df_test['lambda_home'] = lgb_home.predict(df_test[features])
        df_test['lambda_away'] = lgb_away.predict(df_test[features])
        
        if 'group' not in df_test.columns:
            df_test['group'] = [chr(65 + (i // 6)) for i in range(len(df_test))]

        standings = {}
        for idx, row in df_test.iterrows():
            _, _, _, pred_h, pred_a = simulate_match_deterministic(row['lambda_home'], row['lambda_away'])
            grp = row['group']
            t1, t2 = row['home_team'], row['away_team']
            
            for t in (t1, t2):
                if t not in standings: standings[t] = {'group': grp, 'pts': 0, 'gd': 0, 'gf': 0}
            
            if pred_h > pred_a: standings[t1]['pts'] += 3
            elif pred_h < pred_a: standings[t2]['pts'] += 3
            else:
                standings[t1]['pts'] += 1; standings[t2]['pts'] += 1
                
            standings[t1]['gd'] += (pred_h - pred_a); standings[t2]['gd'] += (pred_a - pred_h)
            standings[t1]['gf'] += pred_h; standings[t2]['gf'] += pred_a

        df_standings = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index': 'team'})
        df_standings = df_standings.sort_values(by=['group', 'pts', 'gd', 'gf'], ascending=[True, False, False, False])

        # 3. 32강 진출팀 자동 매핑 (1, 2위 고정 분배)
        r32_teams = {}
        third_place_pool = []

        for grp, group_df in df_standings.groupby('group'):
            group_df = group_df.reset_index(drop=True)
            if len(group_df) >= 1: r32_teams[f'1{grp}'] = group_df.loc[0, 'team']
            if len(group_df) >= 2: r32_teams[f'2{grp}'] = group_df.loc[1, 'team']
            if len(group_df) >= 3: third_place_pool.append(group_df.loc[2])

        # 조 3위 경쟁 순위 정렬 및 와일드카드 선별
        df_3rd = pd.DataFrame(third_place_pool).sort_values(by=['pts', 'gd', 'gf'], ascending=False).reset_index(drop=True)
        qualified_3rd_groups = sorted(list(df_3rd.loc[:7, 'group']))
        qualified_3rd_str = "".join(qualified_3rd_groups)

        # 규칙 테이블 조회
        df_rules = pd.read_csv(RULES_FILE)
        matched_rule = df_rules[df_rules['option'] == qualified_3rd_str]
        if matched_rule.empty: matched_rule = df_rules.iloc[0:1]

        # [수정 포인트] IndexError 원천 방지용 예외 처리 조건 추가 (Fallback 보정)
        slots = ['1A', '1B', '1D', '1E', '1G', '1I', '1K', '1L']
        for idx_slot, slot in enumerate(slots):
            target_3rd_group = matched_rule[slot].values[0].replace('3', '')
            match_subset = df_3rd[df_3rd['group'] == target_3rd_group]
            
            if not match_subset.empty:
                r32_teams[f'3{slot[-1]}'] = match_subset['team'].values[0]
            else:
                # 미스매칭 시 조 3위 순위표상의 최상위권 팀 순차 진출 유도
                backup_team = df_3rd.loc[min(idx_slot, len(df_3rd)-1), 'team']
                r32_teams[f'3{slot[-1]}'] = backup_team

        # 빈 슬롯 채우기용 더미 방어코드
        all_necessary_slots = ['1A','1B','1C','1D','1E','1F','1G','1H','1I','1J','1K','1L',
                               '2A','2B','2C','2D','2E','2F','2G','2H','2I','2J','2K','2L',
                               '3A','3B','3D','3E','3G','3I','3K','3L']
        for s in all_necessary_slots:
            if s not in r32_teams:
                # 대진에 결손이 생기면 순위표 1위 팀들을 순환 배정하여 시뮬레이터 중단 차단
                r32_teams[s] = df_standings['team'].iloc[hash(s)%len(df_standings)]

        # 4. 단판 토너먼트 시뮬레이션 매처
        def simulate_knockout(teamA, teamB):
            row_feat = pd.DataFrame([{
                'home_elo': elo_system.get_rating(teamA), 'away_elo': elo_system.get_rating(teamB),
                'elo_diff': elo_system.get_rating(teamA) - elo_system.get_rating(teamB), 'is_neutral': 1,
                'h_form': sum(team_stats[teamA]['pts'][-5:]) if teamA in team_stats else 0,
                'a_form': sum(team_stats[teamB]['pts'][-5:]) if teamB in team_stats else 0,
            }])
            for col in features:
                if col not in row_feat.columns: row_feat[col] = 1.0
            
            l_h = lgb_home.predict(row_feat[features])[0]
            l_y = lgb_away.predict(row_feat[features])[0]
            p_h, _, p_a, s_h, s_a = simulate_match_deterministic(l_h, l_y)
            
            if s_h == s_a: return teamA if p_h >= p_a else teamB
            return teamA if s_h > s_a else teamB

        # 5. FIFA 공식 토너먼트 대진 트리 가동
        bracket_32 = [
            (r32_teams['1A'], r32_teams['3I']), (r32_teams['1B'], r32_teams['3E']),
            (r32_teams['1C'], r32_teams['2D']), (r32_teams['1D'], r32_teams['3G']),
            (r32_teams['1E'], r32_teams['3L']), (r32_teams['1F'], r32_teams['2E']),
            (r32_teams['1G'], r32_teams['3A']), (r32_teams['1H'], r32_teams['2G']),
            (r32_teams['1I'], r32_teams['3B']), (r32_teams['1J'], r32_teams['2I']),
            (r32_teams['1K'], r32_teams['3K']), (r32_teams['1L'], r32_teams['3D']),
            (r32_teams['2A'], r32_teams['2C']), (r32_teams['2B'], r32_teams['2F']),
            (r32_teams['2D'], r32_teams['2G']), (r32_teams['2E'], r32_teams['2H'])
        ]

        # 각 단계별 서바이벌 연산
        bracket_16 = [simulate_knockout(m[0], m[1]) for m in bracket_32]
        bracket_8 = [simulate_knockout(bracket_16[i], bracket_16[i+1]) for i in range(0, len(bracket_16), 2)]
        bracket_4 = [simulate_knockout(bracket_8[i], bracket_8[i+1]) for i in range(0, len(bracket_8), 2)]
        finalists = [simulate_knockout(bracket_4[i], bracket_4[i+1]) for i in range(0, len(bracket_4), 2)]
        champion = simulate_knockout(finalists[0], finalists[1])

        print(f"🏅 [{filename}] 예측 최종 우승국: ✨ {champion} ✨")

    print("\n" + "="*60)
    print("🎉 8개 고도화 실험 조합 검증 및 안전한 우승국 연동 파이프라인 완수!")

🚀 월드컵 고도화 이론 8종 세트 및 토너먼트 시뮬레이션 시작...

🧪 [실험 시작] 8개의 조합별 시뮬레이션을 순차적으로 구동합니다.

⚙️ 진행 중인 조합: Dynamic K(X) | Adj Form(X) | EMA(X)
🏅 [sub_DynK_X_AdjF_X_EMA_X.csv] 예측 최종 우승국: ✨ Argentina ✨

⚙️ 진행 중인 조합: Dynamic K(X) | Adj Form(X) | EMA(O)
🏅 [sub_DynK_X_AdjF_X_EMA_O.csv] 예측 최종 우승국: ✨ Spain ✨

⚙️ 진행 중인 조합: Dynamic K(X) | Adj Form(O) | EMA(X)
🏅 [sub_DynK_X_AdjF_O_EMA_X.csv] 예측 최종 우승국: ✨ Argentina ✨

⚙️ 진행 중인 조합: Dynamic K(X) | Adj Form(O) | EMA(O)
🏅 [sub_DynK_X_AdjF_O_EMA_O.csv] 예측 최종 우승국: ✨ Spain ✨

⚙️ 진행 중인 조합: Dynamic K(O) | Adj Form(X) | EMA(X)
🏅 [sub_DynK_O_AdjF_X_EMA_X.csv] 예측 최종 우승국: ✨ Argentina ✨

⚙️ 진행 중인 조합: Dynamic K(O) | Adj Form(X) | EMA(O)
🏅 [sub_DynK_O_AdjF_X_EMA_O.csv] 예측 최종 우승국: ✨ France ✨

⚙️ 진행 중인 조합: Dynamic K(O) | Adj Form(O) | EMA(X)
🏅 [sub_DynK_O_AdjF_O_EMA_X.csv] 예측 최종 우승국: ✨ Argentina ✨

⚙️ 진행 중인 조합: Dynamic K(O) | Adj Form(O) | EMA(O)
🏅 [sub_DynK_O_AdjF_O_EMA_O.csv] 예측 최종 우승국: ✨ Spain ✨

🎉 8개 고도화 실험 조합 검증 및 안전한 우승국 연동 파이프라인 완수!
